# Chapter 5 - Applications of the Derivative
*Analytic Geometry and Calculus with Python* (4th edition), S. E. Moore.

Run each cell with **Shift + Enter**. All the libraries used (NumPy, SymPy, Matplotlib, SciPy) are pre-installed on Google Colab.

## Listing 5.1: Tangent and normal lines to a curve

In [ ]:
# Listing 5.1: Tangent and normal lines to a curve
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

x = sp.symbols("x")
f = x**2 + 4*x - 3
a = 1
m = sp.diff(f, x).subs(x, a)              # gradient of the tangent
b = f.subs(x, a)
tangent = sp.expand(b + m*(x - a))
normal = sp.expand(b - (x - a)/m)
print("point:", (a, b), " gradient f'(1) =", m)
print("tangent: y =", tangent)
print("normal : y =", normal, "   i.e. x + 6y = 13")

X = np.linspace(-3, 3, 200)
F, T, N = (sp.lambdify(x, e) for e in (f, tangent, normal))
plt.figure(figsize=(4.8, 4.8))
plt.plot(X, F(X), "k", lw=2, label="$y=x^2+4x-3$")
plt.plot(X, T(X), "C3", label="tangent")
plt.plot(X, N(X), "C0--", label="normal")
plt.plot(a, b, "ko"); plt.ylim(-8, 12)
plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.show()


## Listing 5.2: Related rates - differentiate a relation with respect to time

In [ ]:
# Listing 5.2: Related rates - differentiate a relation with respect to time
import sympy as sp

t = sp.symbols("t")
r = sp.Function("r")(t)

# (a) Balloon: V = (4/3) pi r^3, dV/dt = 100 cm^3/s. Find dr/dt when r = 25 cm.
V = sp.Rational(4, 3)*sp.pi*r**3
dVdt = sp.diff(V, t)                                  # 4 pi r^2 dr/dt
drdt = sp.solve(sp.Eq(dVdt, 100), sp.diff(r, t))[0]
print("balloon: dr/dt =", sp.simplify(drdt.subs(r, 25)), "≈", float(drdt.subs(r, 25)), "cm/s")

# (b) An oil slick near an offshore field spreads in a circle. Its radius grows at
#     0.5 m/min. How fast is its AREA growing when the radius is 300 m?
A = sp.pi*r**2
dAdt = sp.diff(A, t).subs(sp.diff(r, t), sp.Rational(1, 2)).subs(r, 300)
print("oil slick: dA/dt =", dAdt, "≈", round(float(dAdt), 1), "m^2/min")


## Listing 5.3: When is the quickest (and slowest) time to take a Bolt across Accra?

In [ ]:
# Listing 5.3: When is the quickest (and slowest) time to take a Bolt across Accra?
# HYPOTHETICAL model of the trip time from Madina to Accra Central:
#   T(t) = 0.4 t^3 - 5.4 t^2 + 16.8 t + 35  minutes,
#   t = hours after 6:00 am, 0 <= t <= 9 (i.e. 6:00 am to 3:00 pm)
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

t = sp.symbols("t")
T = sp.Rational(2, 5)*t**3 - sp.Rational(27, 5)*t**2 + sp.Rational(84, 5)*t + 35
dT = sp.factor(sp.diff(T, t))
print("T'(t) =", dT)
crit = sp.solve(sp.diff(T, t), t)
print("critical numbers:", crit)

def clock(h):
    hh = 6 + int(h); mm = round(60*(h - int(h)))
    return f"{hh % 12 or 12}:{mm:02d} {'am' if hh < 12 else 'pm'}"

candidates = [0] + crit + [9]          # closed interval method
for c in candidates:
    print(f"t = {c} ({clock(float(c)):>8}):  T = {float(T.subs(t, c)):.1f} min")
best = min(candidates, key=lambda c: T.subs(t, c))
worst = max(candidates, key=lambda c: T.subs(t, c))
print("quickest trip at", clock(float(best)), "; slowest at", clock(float(worst)))

tt = np.linspace(0, 9, 300)
Tf = sp.lambdify(t, T)
plt.figure(figsize=(6, 3.4))
plt.plot(6 + tt, Tf(tt))
for c in candidates:
    plt.plot(6 + float(c), float(T.subs(t, c)), "o")
plt.xlabel("departure time (hour of day)"); plt.ylabel("trip time T (min)")
plt.grid(alpha=0.3); plt.show()


## Listing 5.4: From trip data to a model - Bolt trips UCC -> Kotokuraba (illustrative data)

In [ ]:
# Listing 5.4: From trip data to a model - Bolt trips UCC -> Kotokuraba (illustrative data)
import numpy as np
import matplotlib.pyplot as plt

# departure time (hours after 6:00 am) and observed trip time (minutes)
dep = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
mins = np.array([14, 17, 22, 25, 26, 24, 21, 17, 15, 15, 17, 21])

coef = np.polyfit(dep, mins, 3)          # best-fitting cubic (least squares)
T = np.poly1d(coef)
dT = T.deriv()                           # derivative of the polynomial
print("model:  T(t) =", np.round(coef, 3))
crit = dT.roots
print("critical numbers:", np.round(crit, 3))
for c in crit:
    kind = "maximum" if T.deriv(2)(c) < 0 else "minimum"
    h = 6 + c
    print(f"local {kind} at about {int(h)}:{int(60*(h % 1)):02d}, T = {T(c):.1f} min")

tt = np.linspace(0, 8, 200)
plt.figure(figsize=(6, 3.4))
plt.plot(6 + dep, mins, "o", label="observed trips")
plt.plot(6 + tt, T(tt), label="cubic model")
plt.xlabel("departure time (hour of day)"); plt.ylabel("minutes")
plt.grid(alpha=0.3); plt.legend(); plt.show()


## Listing 5.5: The first derivative test for f(x) = x^(1/3) (8 - x)

In [ ]:
# Listing 5.5: The first derivative test for f(x) = x^(1/3) (8 - x)
import sympy as sp

x = sp.symbols("x", positive=True)       # differentiate where x > 0 ...
f = x**sp.Rational(1, 3) * (8 - x)
df = sp.factor(sp.diff(f, x))
print("f'(x) =", df)                     # ... the same formula holds for x < 0

def fprime(v):                           # real cube roots, valid for negative v too
    return (8 - 4*v) / (3 * sp.real_root(v, 3)**2)

print("f'(x) = 0 at x =", sp.solve(8 - 4*x, x), "; f'(x) undefined at x = 0")
for lo, hi, test in [("-oo", 0, -1), (0, 2, 1), (2, "oo", 3)]:
    s = fprime(test)
    print(f"on ({lo}, {hi}):  f'({test}) = {float(s):+.3f}  ->",
          "increasing" if s > 0 else "decreasing")
print("local maximum f(2) =", f.subs(x, 2), "≈", round(float(f.subs(x, 2)), 3))


## Listing 5.6: Curve sketching with SymPy: f(x) = 12 + 2x^2 - x^4

In [ ]:
# Listing 5.6: Curve sketching with SymPy: f(x) = 12 + 2x^2 - x^4
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

x = sp.symbols("x", real=True)
f = 12 + 2*x**2 - x**4
f1, f2 = sp.diff(f, x), sp.diff(f, x, 2)

for c in sp.solve(f1, x):
    test = f2.subs(x, c)
    kind = "local max" if test < 0 else ("local min" if test > 0 else "test fails")
    print(f"critical x = {c}: f'' = {test} -> {kind}, f = {f.subs(x, c)}")
for p in sp.solve(f2, x):
    print(f"possible inflection at x = {p}, f = {f.subs(x, p)}")

X = np.linspace(-2, 2, 400)
F = sp.lambdify(x, f)
plt.figure(figsize=(5.5, 3.6))
plt.plot(X, F(X), "k")
for c in sp.solve(f1, x):
    plt.plot(float(c), float(f.subs(x, c)), "ro")
for p in sp.solve(f2, x):
    plt.plot(float(p), float(f.subs(x, p)), "gs")
plt.ylim(4, 14); plt.grid(alpha=0.3)
plt.title("red: extrema, green: inflection points")
plt.show()


## Listing 5.7: The inflection point of an S-shaped (logistic) adoption curve

In [ ]:
# Listing 5.7: The inflection point of an S-shaped (logistic) adoption curve
# ILLUSTRATIVE model of active mobile-money accounts in a country (millions):
#   N(t) = 40 / (1 + e^(-0.6 (t - 8))),   t = years after 2010
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

t = sp.symbols("t", real=True)
N = 40 / (1 + sp.exp(-sp.Rational(3, 5)*(t - 8)))
N1, N2 = sp.diff(N, t), sp.diff(N, t, 2)
tp = sp.solve(sp.simplify(N2), t)
print("N''(t) = 0 at t =", tp, "-> year", 2010 + tp[0])
print("N at inflection =", N.subs(t, tp[0]), "million (half of 40)")
print("fastest growth N'(8) =", N1.subs(t, 8), "million per year")

T = np.linspace(0, 16, 300)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(2010 + T, sp.lambdify(t, N)(T)); ax[0].set_title("N(t)")
ax[1].plot(2010 + T, sp.lambdify(t, N1)(T), "C1"); ax[1].set_title("N'(t)")
for a in ax:
    a.axvline(2018, ls="--", color="gray"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Listing 5.8: Three optimisation problems solved with SymPy

In [ ]:
# Listing 5.8: Three optimisation problems solved with SymPy
import sympy as sp

x, r, p = sp.symbols("x r p", positive=True)

# (a) Farm beside the Pra River: 2400 m of fencing, no fence along the river.
A = x*(2400 - 2*x)
xs = sp.solve(sp.diff(A, x), x)[0]
print(f"(a) x = {xs} m, y = {2400 - 2*xs} m, area = {A.subs(x, xs)} m^2")

# (b) A closed tin can holding 400 cm^3 using the least metal.
h = 400 / (sp.pi*r**2)                        # from the volume constraint
S = 2*sp.pi*r**2 + 2*sp.pi*r*h                # surface area
rs = sp.solve(sp.diff(S, r), r)[0]
print(f"(b) r = {float(rs):.3f} cm, h = {float(h.subs(r, rs)):.3f} cm,"
      f" h/r = {sp.simplify(h.subs(r, rs)/rs)}, S = {float(S.subs(r, rs)):.1f} cm^2")

# (c) A waakye seller sells q = 400 - 10p plates a day at price p (GH¢). Max revenue?
R = p*(400 - 10*p)
ps = sp.solve(sp.diff(R, p), p)[0]
print(f"(c) price p = GH¢ {ps}, plates sold = {400 - 10*ps}, revenue = GH¢ {R.subs(p, ps)}")


## Listing 5.9: The cheapest cruising speed for a driver on the Accra-Cape Coast highway

In [ ]:
# Listing 5.9: The cheapest cruising speed for a driver on the Accra-Cape Coast highway
# Illustrative cost per km at speed v km/h:
#   C(v) = 0.6 + 0.01 v + 40 / v   GH¢/km
#   (fixed costs + fuel, which rises with speed + driver's time at GH¢ 40/hour)
import numpy as np
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

C = lambda v: 0.6 + 0.01*v + 40/v

# Calculus: C'(v) = 0.01 - 40/v^2 = 0  ->  v = sqrt(4000)
v_star = np.sqrt(40/0.01)
print(f"calculus : v* = {v_star:.2f} km/h,  C(v*) = GH¢ {C(v_star):.3f} per km")

# Numerical check with SciPy
res = minimize_scalar(C, bounds=(20, 120), method="bounded")
print(f"scipy    : v* = {res.x:.2f} km/h,  C(v*) = GH¢ {res.fun:.3f} per km")
print(f"cost of the 145 km trip at 63 km/h: GH¢ {145*C(63):.2f};  at 100 km/h: GH¢ {145*C(100):.2f}")

v = np.linspace(20, 120, 200)
plt.figure(figsize=(5.5, 3.3))
plt.plot(v, C(v)); plt.plot(v_star, C(v_star), "ro")
plt.xlabel("speed v (km/h)"); plt.ylabel("cost (GH¢ per km)"); plt.grid(alpha=0.3)
plt.show()


## Listing 5.10: Newton's method x_{n+1} = x_n - f(x_n)/f'(x_n)

In [ ]:
# Listing 5.10: Newton's method x_{n+1} = x_n - f(x_n)/f'(x_n)
import math

def newton(f, df, x0, tol=1e-12, max_steps=50, show=False):
    x = x0
    for n in range(1, max_steps + 1):
        x_new = x - f(x)/df(x)
        if show:
            print(f"  step {n}: x = {x_new:.15f}")
        if abs(x_new - x) < tol:
            return x_new, n
        x = x_new
    raise RuntimeError("Newton's method did not converge")

# (a) Heron of Alexandria's square root of 2:  f(x) = x^2 - 2
print("sqrt(2):")
root, n = newton(lambda x: x*x - 2, lambda x: 2*x, 1.0, show=True)
print(f"  math.sqrt(2) = {math.sqrt(2):.15f}")

# (b) x^3 - x - 2 = 0 (compare the bisection method of Listing 3.9: 20 steps)
root, n = newton(lambda x: x**3 - x - 2, lambda x: 3*x**2 - 1, 1.5)
print(f"x^3 - x - 2 = 0:  x = {root:.10f} after {n} steps")

# (c) cos x = x
root, n = newton(lambda x: math.cos(x) - x, lambda x: -math.sin(x) - 1, 1.0)
print(f"cos x = x:        x = {root:.10f} after {n} steps")


## Listing 5.11: L'Hôpital's rule checked with SymPy

In [ ]:
# Listing 5.11: L'Hôpital's rule checked with SymPy
import sympy as sp

x = sp.symbols("x")
cases = [
    ((sp.exp(x) - 1 - x), x**2, 0),
    (sp.log(x), x - 1, 1),
    (x - sp.sin(x), x**3, 0),
    (sp.log(x), x, sp.oo),
    (x**2, sp.exp(x), sp.oo),
]
for f, g, a in cases:
    ratio_of_derivs = sp.diff(f, x) / sp.diff(g, x)
    print(f"lim {str(f/g):<22} x->{str(a):<3} = {str(sp.limit(f/g, x, a)):<4}"
          f"  (f'/g' -> {sp.limit(ratio_of_derivs, x, a)})")
